<a href="https://colab.research.google.com/github/NazHub1993/ML_Notebooks/blob/main/Handling_Missing_Outlier_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [259]:
import pandas as pd
import numpy as np

data = {
    "employee_id": range(1, 16),
    "age": [22, 25, 28, np.nan, 35, 29, 31, 120, 26, 24, np.nan, 42, 30, 27, 150],
    "salary": [25000, 30000, np.nan, 35000, 40000, 38000, 42000,
               45000, np.nan, 32000, 36000, 500000, 41000, 39000, 28000],
    "experience": [0, 2, 4, 6, np.nan, 5, 7, 8, 3, 1, 4, 20, np.nan, 3, 2],
    "department": ["IT", "HR", "IT", "Finance", np.nan, "IT", "HR",
                   "IT", "Finance", "HR", "IT", "Finance", "IT", np.nan, "HR"],
    "performance_score": [70, 75, 80, np.nan, 85, 78, 90, 95,
                          82, 76, np.nan, 98, 88, 79, 150]
}

df = pd.DataFrame(data)

print(df)

    employee_id    age    salary  experience department  performance_score
0             1   22.0   25000.0         0.0         IT               70.0
1             2   25.0   30000.0         2.0         HR               75.0
2             3   28.0       NaN         4.0         IT               80.0
3             4    NaN   35000.0         6.0    Finance                NaN
4             5   35.0   40000.0         NaN        NaN               85.0
5             6   29.0   38000.0         5.0         IT               78.0
6             7   31.0   42000.0         7.0         HR               90.0
7             8  120.0   45000.0         8.0         IT               95.0
8             9   26.0       NaN         3.0    Finance               82.0
9            10   24.0   32000.0         1.0         HR               76.0
10           11    NaN   36000.0         4.0         IT                NaN
11           12   42.0  500000.0        20.0    Finance               98.0
12           13   30.0   

In [260]:
df.isnull().sum()

,0
employee_id,0
age,2
salary,2
experience,2
department,2
performance_score,2


In [261]:
x=df.drop("performance_score",axis=1)
y=df["performance_score"]

In [262]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

#Let's try to find the columns that have missing values

In [263]:
x_train.isnull().sum()

,0
employee_id,0
age,2
salary,2
experience,2
department,2


In [264]:
x_train.drop("employee_id",axis=1,inplace=True)
x_test.drop("employee_id",axis=1,inplace=True)

In [265]:
x_train.shape

(12, 4)

In [266]:
x_test.shape

(3, 4)

In [267]:
numerical_cols=x_train.select_dtypes(include=np.number).columns
categorical_cols=x_train.select_dtypes(exclude=np.number).columns

In [268]:
numerical_cols

Index(['age', 'salary', 'experience'], dtype='object')

In [269]:
categorical_cols

Index(['department'], dtype='object')

#Imputation of missing values using median for numerical_cols and mode for categorical_cols

In [270]:
x_train[numerical_cols]=x_train[numerical_cols].fillna(x_train[numerical_cols].median())
x_test[numerical_cols]=x_test[numerical_cols].fillna(x_train[numerical_cols].median())

In [271]:
x_train.isnull().sum()

,0
age,0
salary,0
experience,0
department,2


In [272]:
x_train[categorical_cols].mode()

,department
0,IT


In [273]:
x_train

,age,salary,experience,department
13,27.0,39000.0,3.0,NaN
5,29.0,38000.0,5.0,IT
8,26.0,38500.0,3.0,Finance
2,28.0,38500.0,4.0,IT
1,25.0,30000.0,2.0,HR
14,150.0,28000.0,2.0,HR
4,35.0,40000.0,4.0,NaN
7,120.0,45000.0,8.0,IT
10,29.5,36000.0,4.0,IT
12,30.0,41000.0,4.0,IT


In [274]:
x_train[categorical_cols]=x_train[categorical_cols].fillna(x_train[categorical_cols].mode().iloc[0])
x_test[categorical_cols]=x_test[categorical_cols].fillna(x_train[categorical_cols].mode().iloc[0])

In [275]:
x_train.isnull().sum()

,0
age,0
salary,0
experience,0
department,0


In [276]:
x_train

,age,salary,experience,department
13,27.0,39000.0,3.0,IT
5,29.0,38000.0,5.0,IT
8,26.0,38500.0,3.0,Finance
2,28.0,38500.0,4.0,IT
1,25.0,30000.0,2.0,HR
14,150.0,28000.0,2.0,HR
4,35.0,40000.0,4.0,IT
7,120.0,45000.0,8.0,IT
10,29.5,36000.0,4.0,IT
12,30.0,41000.0,4.0,IT


In [277]:
x_train.isnull().sum()

,0
age,0
salary,0
experience,0
department,0


#How to define the outliers for each and evry column using a function?

In [278]:
type(x_train)

pandas.core.frame.DataFrame

In [279]:
def find_outliers(df,column):

  q1=df[column].quantile(0.25)
  q3=df[column].quantile(0.75)

  IQR=q3-q1

  lower_bound=q1-(1.5*IQR)
  upper_bound=q3+(1.5*IQR)

  outliers=df[(df[column]<lower_bound) | (df[column]>upper_bound)]

  return outliers




In [280]:
find_outliers(x_train,"age")


,age,salary,experience,department
14,150.0,28000.0,2.0,HR
7,120.0,45000.0,8.0,IT


In [281]:
find_outliers(x_train,"salary")

,age,salary,experience,department
14,150.0,28000.0,2.0,HR


In [282]:
find_outliers(x_train,"experience")

,age,salary,experience,department


In [283]:
def cap_outliers(df1,df2,column):

  q1=df1[column].quantile(0.25)
  q3=df1[column].quantile(0.75)

  IQR=q3-q1

  lower_bound=q1-(1.5*IQR)
  upper_bound=q3+(1.5*IQR)

  df1[column]=df1[column].clip(
      lower=lower_bound,
      upper=upper_bound
  )
  df2[column]=df2[column].clip(
      lower=lower_bound,
      upper=upper_bound
  )

  return df1,df2


In [284]:
x_train,x_test=cap_outliers(x_train,x_test,"age")

In [285]:
x_train,x_test=cap_outliers(x_train,x_test,"salary")

#Time to scale the numerical features

In [286]:
from sklearn.preprocessing import StandardScaler
sc=StandardScaler()

x_train[numerical_cols]=sc.fit_transform(x_train[numerical_cols])
x_test[numerical_cols]=sc.transform(x_test[numerical_cols])


In [287]:
x_train

,age,salary,experience,department
13,-0.835859,0.300552,-0.742781,IT
5,-0.366605,0.075138,0.371391,IT
8,-1.070486,0.187845,-0.742781,Finance
2,-0.601232,0.187845,-0.185695,IT
1,-1.305113,-1.728176,-1.299867,HR
14,1.833023,-1.953590,-1.299867,HR
4,1.041157,0.525967,-0.185695,IT
7,1.833023,1.653038,2.042649,IT
10,-0.249291,-0.375690,-0.185695,IT
12,-0.131978,0.751381,-0.185695,IT


In [288]:
from sklearn.preprocessing import OneHotEncoder
encoder=OneHotEncoder(handle_unknown="ignore",
    sparse_output=False,drop="first")


In [289]:
train_encoded=encoder.fit_transform(x_train[categorical_cols])
test_encoded=encoder.transform(x_test[categorical_cols])


In [290]:
train_encoded

array([[0., 1.],
       [0., 1.],
       [0., 0.],
       [0., 1.],
       [1., 0.],
       [1., 0.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [0., 1.],
       [0., 0.],
       [1., 0.]])

In [291]:
test_encoded

array([[1., 0.],
       [0., 0.],
       [0., 1.]])

In [292]:
type(train_encoded)


numpy.ndarray

In [293]:
encoded_columns=encoder.get_feature_names_out(categorical_cols)

In [294]:
print(encoded_columns)

['department_HR' 'department_IT']


In [295]:
train_encoded_df=pd.DataFrame(train_encoded,columns=encoded_columns,index=x_train.index)
test_encoded_df=pd.DataFrame(test_encoded,columns=encoded_columns,index=x_test.index)

In [296]:
train_encoded_df.head()

,department_HR,department_IT
13,0.0,1.0
5,0.0,1.0
8,0.0,0.0
2,0.0,1.0
1,1.0,0.0


In [297]:
x_train=x_train.drop(categorical_cols,axis=1)
x_test=x_test.drop(categorical_cols,axis=1)

In [298]:
x_train_final=pd.concat([x_train,train_encoded_df],axis=1)
x_test_final=pd.concat([x_test,test_encoded_df],axis=1)

In [299]:
x_train_final

,age,salary,experience,department_HR,department_IT
13,-0.835859,0.300552,-0.742781,0.0,1.0
5,-0.366605,0.075138,0.371391,0.0,1.0
8,-1.070486,0.187845,-0.742781,0.0,0.0
2,-0.601232,0.187845,-0.185695,0.0,1.0
1,-1.305113,-1.728176,-1.299867,1.0,0.0
14,1.833023,-1.953590,-1.299867,1.0,0.0
4,1.041157,0.525967,-0.185695,0.0,1.0
7,1.833023,1.653038,2.042649,0.0,1.0
10,-0.249291,-0.375690,-0.185695,0.0,1.0
12,-0.131978,0.751381,-0.185695,0.0,1.0


#Now coming to the most important step which is creating the pipeline

In [300]:
from sklearn.base import BaseEstimator, TransformerMixin
import pandas as pd


class IQRClipper(BaseEstimator, TransformerMixin):

    def __init__(self, factor=1.5):
        self.factor = factor

    def fit(self, X, y=None):
        X = pd.DataFrame(X)

        Q1 = X.quantile(0.25)
        Q3 = X.quantile(0.75)

        IQR = Q3 - Q1

        self.lower_bound_ = Q1 - self.factor * IQR
        self.upper_bound_ = Q3 + self.factor * IQR

        return self

    def transform(self, X):
        X = pd.DataFrame(X).copy()

        for column in X.columns:
            X[column] = X[column].clip(
                lower=self.lower_bound_[column],
                upper=self.upper_bound_[column]
            )

        return X

In [301]:
from sklearn.pipeline import Pipeline

In [302]:
df_final=pd.DataFrame(data)

In [303]:
df_final.head()

,employee_id,age,salary,experience,department,performance_score
0,1,22.0,25000.0,0.0,IT,70.0
1,2,25.0,30000.0,2.0,HR,75.0
2,3,28.0,NaN,4.0,IT,80.0
3,4,NaN,35000.0,6.0,Finance,NaN
4,5,35.0,40000.0,NaN,NaN,85.0


In [304]:
df_final=df_final.drop("employee_id",axis=1)

In [305]:
df_final = df_final.dropna(
    subset=["performance_score"]
)

In [306]:
x=df_final.drop("performance_score",axis=1)
y=df_final["performance_score"]

In [307]:
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

In [308]:
from sklearn.impute import SimpleImputer

In [309]:
num_pipeline=Pipeline(
    [
        ("imputer",SimpleImputer(strategy="median")),
        ("outlier",IQRClipper(factor=1.5)),
        ("scaler",StandardScaler())
    ]
)


In [310]:
cat_pipeline=Pipeline(
    [
        ("imputer",SimpleImputer(strategy="most_frequent")),
        ("encoder",OneHotEncoder(handle_unknown='ignore',drop='first'))
    ]
)

In [311]:
from sklearn.compose import ColumnTransformer

In [312]:
preprocessor=ColumnTransformer(
    [
        ('num_pipeline',num_pipeline,numerical_cols),
        ('cat_pipeline',cat_pipeline,categorical_cols)
    ]
)

In [313]:
from sklearn.model_selection import RandomizedSearchCV

In [321]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression

In [322]:
pipe=Pipeline(
    [
        ("preprocessor",preprocessor),
        ("model",LinearRegression())
    ]
)

In [323]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('outlier',
                                                                   IQRClipper()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['age', 'salary', 'experience'], dtype='object')),
                                                 ('cat_pipeline',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(drop='first',
                                                                                 handle_unknown='ignore'))]),
                                                  Index(['department'], dtype='object'))])),
                ('model', LinearRegression())])

In [324]:
y_pred=pipe.predict(x_test)

In [325]:
from sklearn.metrics import r2_score

In [326]:
from sklearn.metrics import r2_score
r2_score(y_test,y_pred)

0.18681435478124575